In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,precision_score, recall_score, f1_score ,average_precision_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

df = pd.read_csv(r"E:\Work\AI\MAKTAB\HW-CW-S02\Mini_Project_01\data\Cleaned_data.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
x_train,x_test,y_train,y_test=train_test_split(df.iloc[:, :-1],df.iloc[:, -1],stratify=df.iloc[:, -1],test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)


In [11]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42
)

param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05],
    "min_child_weight": [1, 3],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_xg = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1,
    verbose=1
)

grid_xg.fit(x_train, y_train)

print("Best parameters:")
print(grid_xg.best_params_)

print("Best CV F1:")
print(grid_xg.best_score_)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}
Best CV F1:
0.863851898797743


In [12]:
model_xg=grid_xg.best_estimator_
model_xg.fit(x_train,y_train)
y_pred_xg=model_xg.predict(x_test)
print(classification_report(y_test,y_pred_xg))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.93      0.79      0.85        95

    accuracy                           1.00     56746
   macro avg       0.96      0.89      0.93     56746
weighted avg       1.00      1.00      1.00     56746



In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [10, 11,12,13,14],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(x_train, y_train)

print("Best Parameters:")
print(grid_rf.best_params_)

print("\nBest CV F1:")
print(grid_rf.best_score_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Parameters:
{'max_depth': 13, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

Best CV F1:
0.8495134534725141


In [ ]:
model_rf=grid_rf.best_estimator_
model_rf.fit(x_train,y_train)
y_pred_rf=model_rf.predict(x_test)
print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.92      0.73      0.81        95

    accuracy                           1.00     56746
   macro avg       0.96      0.86      0.91     56746
weighted avg       1.00      1.00      1.00     56746



In [26]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_xgb = np.zeros(len(y_train))
oof_rf = np.zeros(len(y_train))
oof_knn = np.zeros(len(y_train))

for fold, (train_idx, val_idx) in enumerate(
    cv.split(x_train, y_train)
):

    x_tr = x_train.iloc[train_idx]
    x_va = x_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_va = y_train.iloc[val_idx]

    neg = (y_tr == 0).sum()
    pos = (y_tr == 1).sum()

    scale_pos_weight = neg / pos

    knn_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=3,
            weights="distance",
            p=1
        ))
    ])

    print(f"Training Fold {fold + 1}/5...")

    model_xg.fit(x_tr, y_tr)
    model_rf.fit(x_tr, y_tr)
    knn_model.fit(x_tr, y_tr)

    oof_xgb[val_idx] = model_xg.predict_proba(x_va)[:, 1]
    oof_rf[val_idx] = model_rf.predict_proba(x_va)[:, 1]
    oof_knn[val_idx] = knn_model.predict_proba(x_va)[:, 1]


best_f1 = -1
best_weights = None
best_threshold = None

best_f1_any = -1
best_weights_any = None
best_threshold_any = None


for w_xgb in np.arange(0.0, 1.01, 0.05):

    for w_rf in np.arange(0.0, 1.01 - w_xgb, 0.05):

        w_knn = 1 - w_xgb - w_rf

        ensemble_prob = (
            w_xgb * oof_xgb +
            w_rf * oof_rf +
            w_knn * oof_knn
        )

        for threshold in np.arange(0.05, 0.96, 0.01):

            y_pred = (
                ensemble_prob >= threshold
            ).astype(int)

            precision = precision_score(
                y_train,
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_train,
                y_pred,
                zero_division=0
            )

            f1 = f1_score(
                y_train,
                y_pred,
                zero_division=0
            )

            if f1 > best_f1_any:

                best_f1_any = f1

                best_weights_any = (
                    w_xgb,
                    w_rf,
                    w_knn
                )

                best_threshold_any = threshold

            if (
                precision >= 0.90
                and recall >= 0.80
                and f1 > best_f1
            ):

                best_f1 = f1

                best_weights = (
                    w_xgb,
                    w_rf,
                    w_knn
                )

                best_threshold = threshold


if best_weights is None:

    print("\nNo combination satisfied:")
    print("Precision >= 0.90")
    print("Recall >= 0.80")

    print("\nUsing best F1 combination instead.")

    best_weights = best_weights_any
    best_threshold = best_threshold_any
    best_f1 = best_f1_any


w_xgb, w_rf, w_knn = best_weights


print("\n==============================")
print("BEST PARAMETERS")
print("==============================")

print("XGBoost:", w_xgb)
print("Random Forest:", w_rf)
print("KNN:", w_knn)
print("Threshold:", best_threshold)
print("OOF F1:", best_f1)


print("\nTraining final models...")




Training Fold 1/5...
Training Fold 2/5...
Training Fold 3/5...
Training Fold 4/5...
Training Fold 5/5...

BEST PARAMETERS
XGBoost: 0.45
Random Forest: 0.45
KNN: 0.10000000000000003
Threshold: 0.5600000000000002
OOF F1: 0.8727272727272727

Training final models...

TEST RESULTS
Threshold: 0.5600000000000002
Precision: 0.972972972972973
Recall: 0.7578947368421053
F1: 0.8520710059171598
PR-AUC: 0.8252745318648874

Confusion Matrix:
[[56649     2]
 [   23    72]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.76      0.85        95

    accuracy                           1.00     56746
   macro avg       0.99      0.88      0.93     56746
weighted avg       1.00      1.00      1.00     56746



In [27]:
model_xg.fit(x_train, y_train)
model_rf.fit(x_train, y_train)

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=3,
        weights="distance",
        p=1
    ))
])

knn_model.fit(x_train, y_train)


xgb_prob = model_xg.predict_proba(x_test)[:, 1]
rf_prob = model_rf.predict_proba(x_test)[:, 1]
knn_prob = knn_model.predict_proba(x_test)[:, 1]


ensemble_prob = (
    w_xgb * xgb_prob +
    w_rf * rf_prob +
    w_knn * knn_prob
)


y_pred = (
    ensemble_prob >= best_threshold
).astype(int)


print("\n==============================")
print("TEST RESULTS")
print("==============================")

print("Threshold:", best_threshold)

print(
    "Precision:",
    precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        ensemble_prob
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


TEST RESULTS
Threshold: 0.5600000000000002
Precision: 0.972972972972973
Recall: 0.7578947368421053
F1: 0.8520710059171598
PR-AUC: 0.8252745318648874

Confusion Matrix:
[[56649     2]
 [   23    72]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.76      0.85        95

    accuracy                           1.00     56746
   macro avg       0.99      0.88      0.93     56746
weighted avg       1.00      1.00      1.00     56746

